# Чистовик: константный прогноз

Заготовка, которая уже проходит весь путь: читает `data/`, считает CV по фолдам, пишет `submission.csv` и `cv.json` и печатает `cv.json` последней строкой. Замените прогноз и разбиение на свои.

In [62]:
import json, sys
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from catboost import CatBoostRegressor

sys.path.insert(0, "data")
from metric_cv import quantile_quality, cv_json

tr = pd.read_csv("data/train.csv")
te = pd.read_csv("data/test.csv")
sample = pd.read_csv("data/sample_submission.csv")
y = tr["daily_orders"].to_numpy(float)
QUANTILES = (0.1, 0.5, 0.9)


X_tr = tr.drop(columns=['id', 'daily_orders'])
X_te = te.drop(columns=['id'])

cat_features = ['city', 'district_type', 'entrance_type']


def predict(X_tr, y_tr, X_val):
    model = CatBoostRegressor(
        loss_function="MultiQuantile:alpha=0.1,0.5,0.9",
        cat_features=cat_features,
        iterations=1000,
        verbose=500,
        thread_count=-1,
        random_state=0
    )
    model.fit(X_tr, y_tr)

    pred = model.predict(X_val)
    pred = np.maximum.accumulate(pred, axis=1)
    return np.clip(pred, 0, 5000)

In [ ]:


# CV: четыре случайных фолда — замените разбиением, которое похоже на тест
splits = list(KFold(4, shuffle=True, random_state=0).split(tr))
oof = np.zeros((len(tr), 3))
for a, b in splits:
    oof[b] = predict(X_tr.iloc[a], y[a], X_tr.iloc[b])
folds = [100 * quantile_quality(y[b], oof[b, 0], oof[b, 1], oof[b, 2], tr["city"].iloc[b]) for _, b in splits]

                                 # последняя строка — для проверки воспроизводимости




0:	learn: 20.7892813	total: 19.1ms	remaining: 19.1s
500:	learn: 3.8890053	total: 6.23s	remaining: 6.21s
999:	learn: 3.3228791	total: 12.6s	remaining: 0us
0:	learn: 21.8131531	total: 10.8ms	remaining: 10.8s
500:	learn: 4.0914586	total: 6.27s	remaining: 6.25s
999:	learn: 3.4447033	total: 12.7s	remaining: 0us
0:	learn: 22.1543983	total: 10.8ms	remaining: 10.8s
500:	learn: 4.1284421	total: 6.35s	remaining: 6.32s
999:	learn: 3.5123695	total: 12.5s	remaining: 0us
0:	learn: 20.9309222	total: 10.3ms	remaining: 10.3s
500:	learn: 4.0126741	total: 6.34s	remaining: 6.32s
999:	learn: 3.3300249	total: 12.8s	remaining: 0us
{"name": "catboost_const_x1.5", "metric": "quantile_quality_x100", "folds": [76.5, 78.01, 77.37, 78.44], "cv": 77.58, "std": 0.73}


In [61]:
cv = cv_json("catboost_multi", [f / 100 for f in folds])   # пишет cv.json
print(json.dumps(cv)) 

{"name": "catboost_multi", "metric": "quantile_quality_x100", "folds": [76.5, 78.01, 77.37, 78.44], "cv": 77.58, "std": 0.73}


In [63]:
# посылка: колонки и строки как в sample_submission.csv, 0 <= p10 <= p50 <= p90 <= 5000
test = np.round(predict(X_tr, y, X_te), 1)
sub = pd.DataFrame({"id": te["id"],
                    "predicted_daily_orders_p10": test[:, 0],
                    "predicted_daily_orders_p50": test[:, 1],
                    "predicted_daily_orders_p90": test[:, 2]})
assert list(sub.columns) == list(sample.columns) and len(sub) == len(sample)
assert set(sub["id"]) == set(sample["id"]) and (sub.iloc[:, 1:].to_numpy() >= 0).all()
sub.to_csv("submission.csv", index=False)



0:	learn: 21.3978822	total: 16.6ms	remaining: 16.6s
500:	learn: 4.1652637	total: 6.7s	remaining: 6.68s
999:	learn: 3.6217411	total: 13.6s	remaining: 0us
